# APUSH LEQ Grader SLM — v1 QLoRA train + base-vs-tuned eval (Colab)

Runs the Day-3 gate on a GPU: train the v1 QLoRA adapter, then eval it on both tracks and put the numbers on the board.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough for a 0.5B model).

Order: check GPU → clone → install → (HF token) → train → eval litmus → eval real → print base-vs-tuned table.

`artifacts/data/` is committed to the repo, so no data regeneration is needed — the clone already has `train_chat.jsonl`, `eval_cases.jsonl`, and `eval_real_cases.jsonl`.

## 1. Confirm the GPU is attached

In [1]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then rerun.'

Thu Jul  9 16:42:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone the repo

Public repo — the committed `artifacts/data/` comes with it. Re-run-safe: pulls latest if the folder already exists.

In [2]:
import os
REPO = 'https://github.com/aryanjverma/slm.git'
if not os.path.isdir('/content/slm'):
    !git clone $REPO /content/slm
else:
    !cd /content/slm && git pull
%cd /content/slm
!wc -l artifacts/data/train_chat.jsonl artifacts/data/eval_cases.jsonl artifacts/data/eval_real_cases.jsonl

Cloning into '/content/slm'...
remote: Enumerating objects: 230, done.
remote: Counting objects: 100% (230/230), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 230 (delta 75), reused 214 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (230/230), 664.69 KiB | 10.90 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/content/slm
   1000 artifacts/data/train_chat.jsonl
    198 artifacts/data/eval_cases.jsonl
     72 artifacts/data/eval_real_cases.jsonl
   1270 total


## 3. Install training extras

Installs the package with the `[train]` extra (unsloth / trl / peft / bitsandbytes). Takes a few minutes; a pip resolver warning about pre-installed Colab packages is normal.

In [11]:
!pip install -q -e ".[train]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for apush-frq-grader-slm (pyproject.toml) ... done


## 4. (Optional) Hugging Face token

Avoids rate-limited base-model downloads. Add a Colab secret named `HF_TOKEN` (🔑 in the left sidebar), or skip this cell to download anonymously.

In [4]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set from Colab secrets.')
except Exception as e:
    print('No HF_TOKEN (downloading anonymously):', e)

No HF_TOKEN (downloading anonymously): Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.


## 5. Train the v1 QLoRA adapter

Defaults: `Qwen/Qwen2.5-0.5B-Instruct`, 300 steps, LoRA rank 16 — a few minutes on a T4. Writes the adapter to `artifacts/models/apush-frq-grader-v1/`.

In [13]:
!python scripts/train_qlora.py \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --data artifacts/data/train_chat.jsonl \
  --output artifacts/models/apush-frq-grader-v1 \
  --max-steps 300

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_lin

## 6. Refresh the deterministic baselines

Regenerates `inflated_prompted_base` and `apush_grader_reference` on the same 198-case litmus set so the tuned model is compared against them on identical inputs. CPU-fast.

In [6]:
!python -m apush_frq_grader_slm.cli.run_eval \
  --eval-path artifacts/data/eval_cases.jsonl \
  --output-dir artifacts/eval

inflated_prompted_base: json=1.00, rubric=0.82, grounding=0.17, robustness=0.93,
total=0.69
apush_grader_reference: json=1.00, rubric=1.00, grounding=1.00, robustness=2.00,
total=1.00


## 6b. Tokenizer deps for eval

The eval script loads the Qwen tokenizer in a fresh subprocess, which needs 
`sentencepiece`/`tiktoken` to build the fast tokenizer. These are now in the 
`[train]` extra, but install them explicitly here so an already-installed runtime is covered.

In [ ]:
!pip install -q sentencepiece tiktoken

## 7. Eval the tuned model — litmus track (the gate)

Runs the tuned adapter over the synthetic contract + adversarial set. This is the base-vs-tuned regression signal.

In [7]:
!python scripts/eval_hf_model.py \
  --model artifacts/models/apush-frq-grader-v1 \
  --model-name apush_frq_grader_v1 \
  --eval-path artifacts/data/eval_cases.jsonl \
  --output-dir artifacts/eval

Traceback (most recent call last):
  File "/content/slm/scripts/eval_hf_model.py", line 107, in <module>
    main()
  File "/content/slm/scripts/eval_hf_model.py", line 58, in main
    model, tokenizer, device = load_model_and_tokenizer(args.model)
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/slm/scripts/eval_hf_model.py", line 37, in load_model_and_tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 817, in from_pretrained
    return tokenizer_class.from_pretrained(pretrained_model_name_or_path, *inputs, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1721, in from_pretrained
    return cls._from_pretrained(
           ^^^^^^^^^^^

## 8. Eval the tuned model — real track (College Board agreement)

Row-score agreement and QWK vs the ~72 real CB essays. Eval only — these never entered training.

In [8]:
!python scripts/eval_hf_model.py \
  --model artifacts/models/apush-frq-grader-v1 \
  --model-name apush_frq_grader_v1 \
  --eval-path artifacts/data/eval_real_cases.jsonl \
  --real-eval \
  --output-dir artifacts/eval

Traceback (most recent call last):
  File "/content/slm/scripts/eval_hf_model.py", line 107, in <module>
    main()
  File "/content/slm/scripts/eval_hf_model.py", line 58, in main
    model, tokenizer, device = load_model_and_tokenizer(args.model)
                               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/slm/scripts/eval_hf_model.py", line 37, in load_model_and_tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/tokenization_auto.py", line 817, in from_pretrained
    return tokenizer_class.from_pretrained(pretrained_model_name_or_path, *inputs, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py", line 1721, in from_pretrained
    return cls._from_pretrained(
           ^^^^^^^^^^^

## 9. Base-vs-tuned summary table

Reads every summary written to `artifacts/eval/` and prints the litmus comparison. The gate is met when `apush_frq_grader_v1` beats `inflated_prompted_base` on grounding and robustness.

In [9]:
import json, glob, os

def load(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

# Litmus summaries: baselines live in summary.jsonl; tuned model in its own *_summary.jsonl.
rows = {}
sp = 'artifacts/eval/summary.jsonl'
if os.path.exists(sp):
    for r in load(sp):
        rows[r['model_name']] = r
for p in glob.glob('artifacts/eval/*_summary.jsonl'):
    if 'slice' in p or 'real' in p or p.endswith('summary.jsonl') and os.path.basename(p) == 'summary.jsonl':
        continue
    for r in load(p):
        rows[r['model_name']] = r

order = ['inflated_prompted_base', 'apush_frq_grader_v1', 'apush_grader_reference']
ranked = [rows[m] for m in order if m in rows] + [r for m, r in rows.items() if m not in order]

hdr = f"{'model':<28} {'n':>4} {'json':>6} {'rubric':>7} {'ground':>7} {'robust':>7} {'total':>6}"
print('LITMUS (198-case synthetic contract + adversarial)')
print(hdr); print('-' * len(hdr))
for r in ranked:
    print(f"{r['model_name']:<28} {r['count']:>4} {r['structured_output_valid_rate']:>6.2f} "
          f"{r['rubric_accuracy_mean']:>7.2f} {r['evidence_grounding_rate']:>7.2f} "
          f"{r['robustness_mean']:>7.2f} {r['total_score_mean']:>6.2f}")

# Real track (tuned model only).
rp = 'artifacts/eval/apush_frq_grader_v1_real_summary.jsonl'
if os.path.exists(rp):
    r = load(rp)[0]
    print('\nREAL (College Board essays)')
    print(f"  cases={r['count']}  json_valid={r['structured_output_valid_rate']:.2f}")
    print(f"  row exact={r['exact_match_rate']:.2f}  row within-1={r['within_one_rate']:.2f}")
    print(f"  total exact={r['total_exact_match_rate']:.2f}  total within-1={r['total_within_one_rate']:.2f}  QWK={r['qwk']}")

LITMUS (198-case synthetic contract + adversarial)
model                           n   json  rubric  ground  robust  total
-----------------------------------------------------------------------
inflated_prompted_base        198   1.00    0.82    0.17    0.93   0.69
apush_grader_reference        198   1.00    1.00    1.00    2.00   1.00


## 10. (Optional) Save the adapter to Google Drive

The adapter is `.gitignore`d and Colab storage is ephemeral. Copy it to Drive to keep it after the runtime disconnects.

In [10]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/slm_models
!cp -r artifacts/models/apush-frq-grader-v1 /content/drive/MyDrive/slm_models/
!ls -la /content/drive/MyDrive/slm_models/apush-frq-grader-v1

ValueError: mount failed